In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# !wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.tgz .
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.gz .
!tar xf spark-3.1.1-bin-hadoop3.2.gz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("PROCESS2") \
        .getOrCreate()

sc = spark.sparkContext
baskets = "/content/drive/MyDrive/MMDS-data/baskets.csv"
ratings2k = "/content/drive/MyDrive/MMDS-data/ratings2k.csv"

**Task 1:**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth

class FPGrowthModel:
    def __init__(self, baskets_path, min_support=0.01, min_confidence=0.000000001):
        self.baskets_path = baskets_path
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.df_baskets = None
        self.model = None

    def discover_baskets(self):
        df = spark.read.csv(self.baskets_path, header=True, inferSchema=True)
        self.df_baskets = df.select('Member_number', 'Date', 'itemDescription') \
                            .groupBy('Member_number', 'Date') \
                            .agg(F.collect_set('itemDescription').alias('Items')) \
                            .orderBy('Date')

    def fit_model(self):
        if self.df_baskets is None:
            raise ValueError("Data has not been prepared.")
        fpGrowth = FPGrowth(itemsCol="Items", minSupport=self.min_support, minConfidence=self.min_confidence)
        self.model = fpGrowth.fit(self.df_baskets)

    def display_frequent_itemsets(self):
        if self.model is None:
            raise ValueError("Model has not been fitted.")
        frequentItemsets = self.model.freqItemsets
        frequentItemsets.show(truncate=False)

    def display_association_rules(self):
        if self.model is None:
            raise ValueError("Model has not been fitted.")
        association_rules = self.model.associationRules
        association_rules.show(truncate=False)

    def apply_model(self):
        if self.model is None:
            raise ValueError("Model has not been fitted.")
        self.model.transform(self.df_baskets).show(truncate=False)

if __name__ == "__main__":
    fp_growth_model = FPGrowthModel(baskets)
    fp_growth_model.discover_baskets()
    fp_growth_model.fit_model()
    fp_growth_model.display_frequent_itemsets()
    fp_growth_model.display_association_rules()
    fp_growth_model.apply_model()

+------------------------------+----+
|items                         |freq|
+------------------------------+----+
|[whole milk]                  |2363|
|[other vegetables]            |1827|
|[other vegetables, whole milk]|222 |
|[rolls/buns]                  |1646|
|[rolls/buns, other vegetables]|158 |
|[rolls/buns, whole milk]      |209 |
|[soda]                        |1453|
|[soda, whole milk]            |174 |
|[yogurt]                      |1285|
|[yogurt, whole milk]          |167 |
|[root vegetables]             |1041|
|[tropical fruit]              |1014|
|[bottled water]               |908 |
|[sausage]                     |903 |
|[citrus fruit]                |795 |
|[pastry]                      |774 |
|[pip fruit]                   |734 |
|[shopping bags]               |712 |
|[canned beer]                 |702 |
|[bottled beer]                |678 |
+------------------------------+----+
only showing top 20 rows

+------------------+------------------+-------------------+---

**Task 2:**

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

class ALSModel:
  def __init__(self):
    self.spark = spark
    self.recommendations = None
    self.user_recommendations = None
    self.model = None

  def load_data(self):
    self.data = self.spark.read.csv(ratings2k, header=True, inferSchema=True)

  def split_data(self, train_ratio=0.8):
    if not self.data:
      raise ValueError("Data must be loaded before splitting.")

    # Split the ratings dataframe into training and test data
    self.train, self.test = self.data.randomSplit([train_ratio, 1-train_ratio], seed=42)

  def train_model(self, rank=10, maxIter=20, userCol="user", itemCol="item", ratingCol="rating"):
    if not self.train:
      raise ValueError("Training data must be available before training the model.")

    # Set the ALS hyperparameters
    self.als = ALS(rank=rank, userCol=userCol, itemCol=itemCol, ratingCol=ratingCol,
                   coldStartStrategy="drop",
                   maxIter=maxIter,
                   regParam=0.1,
                   nonnegative=True)
    # Fit the model to the training data
    self.model = self.als.fit(self.train)

  def evaluate_model(self):
    if not self.model or not self.test:
      raise ValueError("Model and test data must be available for evaluation.")

    # Generate predictions on the test_data
    predictions = self.model.transform(self.test)
    # Evaluate the model
    evaluator = RegressionEvaluator(metricName= "rmse",
                                    labelCol= "rating",
                                    predictionCol= "prediction")
    rmse = evaluator.evaluate(predictions)
    return rmse

  def product_recommendation(self, num_recs):
    if not self.model:
      raise ValueError("Model must be trained before generating recommendations.")

    # Recommend products for all users
    user_recs = self.model.recommendForAllUsers(num_recs)
    self.recommendations = user_recs
    return user_recs

  def user_recommendation(self, num_recs):
    if not self.model:
      raise ValueError("Model must be trained before generating recommendations.")

    # Recommend users for all products
    product_recs = self.model.recommendForAllItems(num_recs)
    self.user_recommendations = product_recs
    return product_recs

  def recommend_for_user(self, user_id):
    recs_for_user = self.recommendations.filter(self.recommendations.user == user_id).collect()
    return recs_for_user

  def print_recommendations(self, num_recs, user_id=None):
    if not self.recommendations:
        self.product_recommendation(num_recs)
    if not self.user_recommendations:
        self.user_recommendation(num_recs)

    if user_id is not None:
        recs = self.recommend_for_user(user_id)
        if recs:
            print(f"Recommendations for user {user_id}:")
            for row in recs:
                items = row.recommendations
                for item in items:
                    print(f"Item: {item['item']}, Rating: {item['rating']:.2f}")
        else:
            print(f"No recommendations found for user {user_id}.")
    else:
        print(f"Top {num_recs} recommendations for all users:")
        self.recommendations.show(truncate=False)
        print(f"Top {num_recs} recommendations for all products:")
        self.user_recommendations.show(truncate=False)

  def run(self):
    self.load_data()
    self.split_data()
    self.train_model()
    rmse = self.evaluate_model()
    print(f"Mean Squared Error (RMSE) on test data: {rmse:.4f}")
    self.print_recommendations(num_recs=10)
    self.print_recommendations(num_recs=10, user_id=20)

recommend_model = ALSModel()
recommend_model.run()

Mean Squared Error (RMSE) on test data: 1.0201
Top 10 recommendations for all users:
+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user|recommendations                                                                                                                                                                  |
+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|31  |[{429, 4.3384223}, {422, 3.615354}, {462, 3.3021786}, {433, 3.2942615}, {459, 3.2765636}, {99, 3.2493207}, {444, 3.2230694}, {176, 3.1962724}, {421, 3.1578393}, {431, 3.151847}]|
|53  |[{36, 5.109202}, {432, 4.8478637}, {123, 4.7809877}, {430, 4.746581}, {437, 4.5547795}, {462, 4.4723935}, {413, 4.430832}, {99, 4.4115767}, {288, 4.358371}, {210, 4.3570